# Layer 4 — Intelligence Engine: Quality Tests

Validates the complete Layer 4 pipeline output (`intelligence_results.csv`) across 12 assertions.
Run all cells top-to-bottom after executing the four Layer 4 scripts in order:
`4.1` → `4.2` → `4.3` → `4.4`

| Script | Output | Rows | Cols | Key additions |
|---|---|---|---|---|
| `4.1_business_impact_quantification.py` | `impact_results.csv` | 181 | 51 | `revenue_at_risk`, `margin_impact`, `customer_impact`, `monthly_shortfall`, `impact_pct_of_plan`, `impact_narrative` |
| `4.2_prioritization engine.py` | `priority_results.csv` | 181 | 59 | `priority_score`, `priority_band`, `priority_rank`, factor columns |
| `4.3_recommendation_engine.py` | `recommendations.csv` | 181 | 68 | `immediate_action`, `short_term_fix`, `preventive_measure`, `playbook_key`, `llm_enhanced` |
| `4.4_intelligence_assembly.py` | `intelligence_results.csv` | 181 | 68 | Validated final Layer 4 output |


In [1]:
import sqlite3
import pandas as pd

df   = pd.read_csv("../data/intelligence_results.csv")
prod = pd.read_csv("../data/products.csv")
AVG_GROSS_MARGIN = prod["gross_margin"].mean()

print(f"intelligence_results.csv loaded: {df.shape[0]} rows x {df.shape[1]} cols")
print(f"avg_gross_margin (from products.csv): {AVG_GROSS_MARGIN:.6f}")

intelligence_results.csv loaded: 181 rows x 68 cols
avg_gross_margin (from products.csv): 0.496782


---
## Test 1 — Shape
The final Layer 4 output must have exactly 181 rows (one per confirmed anomaly) and 68 columns
(45 from `rca_assembly` + 6 impact + 8 priority + 9 recommendation columns).

In [2]:
assert df.shape == (181, 68), f"Expected (181, 68), got {df.shape}"
print(f"PASS  Shape: {df.shape}")

PASS  Shape: (181, 68)


---
## Test 2 — Priority Score Range
`priority_score` is a weighted composite of five factors. All values must be in [0, 1]
with no nulls. The minimum (0.4311) belongs to a near-zero-impact Tier 3 anomaly;
the maximum (0.9911) is the Black Friday revenue spike.

In [3]:
ps = df["priority_score"]

assert ps.notna().all(),           "priority_score contains nulls"
assert ps.between(0, 1).all(),     "priority_score out of [0, 1]"

print(f"PASS  priority_score: {ps.notna().sum()} non-null values in [0, 1]")
print(f"PASS  min = {ps.min():.4f}  max = {ps.max():.4f}  mean = {ps.mean():.4f}")

PASS  priority_score: 181 non-null values in [0, 1]
PASS  min = 0.4311  max = 0.9911  mean = 0.7283


---
## Test 3 — Priority Rank Uniqueness
`priority_rank` assigns every anomaly a unique integer from 1 (most urgent) to 181 (least urgent).
No ties or gaps are allowed — `method='first'` ranking is used in Step 4.2.

In [4]:
pr = df["priority_rank"]

assert pr.nunique() == 181,                  "priority_rank contains duplicates"
assert set(pr) == set(range(1, 182)),        "priority_rank is not a clean 1-181 sequence"

print(f"PASS  priority_rank: {pr.nunique()} unique integers  "
      f"(min={pr.min()}  max={pr.max()})")

PASS  priority_rank: 181 unique integers  (min=1  max=181)


---
## Test 4 — ESCALATE Anomalies → HIGH Priority Band
All 15 HIGH-severity anomalies carry `layer4_priority_flag = ESCALATE` and must score above
the 0.75 threshold into the HIGH priority band. These are the Tier 1 KPIs where all three
detection methods agreed.

In [5]:
esc = df[df["layer4_priority_flag"] == "ESCALATE"]

assert len(esc) == 15,                          f"Expected 15 ESCALATE rows, got {len(esc)}"
assert (esc["priority_band"] == "HIGH").all(),  "Some ESCALATE anomalies not in HIGH band"

print(f"PASS  ESCALATE rows       : {len(esc)} / 15")
print(f"PASS  All in HIGH band    : {( esc['priority_band']=='HIGH').sum()} / {len(esc)}")
print(f"PASS  Score range (ESCALATE): {esc['priority_score'].min():.4f} – {esc['priority_score'].max():.4f}")

PASS  ESCALATE rows       : 15 / 15
PASS  All in HIGH band    : 15 / 15
PASS  Score range (ESCALATE): 0.8330 – 0.9911


---
## Test 5 — SUPPRESSED Routing Integrity
The 6 suppressed anomalies (avg_roas DOWN driven by competitive marketing pressure) must never
receive LLM calls and must retain `escalation_suppressed = True`. HIGH severity anomalies are
never suppressed — this is enforced by Step 3.3.

In [6]:
sup = df[df["layer4_priority_flag"] == "SUPPRESSED"]

assert len(sup) == 6,                          f"Expected 6 SUPPRESSED rows, got {len(sup)}"
assert (~sup["llm_enhanced"]).all(),           "Some SUPPRESSED rows received LLM calls"
assert sup["escalation_suppressed"].all(),     "Some SUPPRESSED rows have escalation_suppressed=False"
assert (sup["severity"] == "MEDIUM").all(),    "Some SUPPRESSED rows are not MEDIUM severity"

print(f"PASS  SUPPRESSED rows          : {len(sup)} / 6")
print(f"PASS  llm_enhanced = False     : {(~sup['llm_enhanced']).sum()} / {len(sup)}")
print(f"PASS  escalation_suppressed    : {sup['escalation_suppressed'].sum()} / {len(sup)}")
print(f"PASS  All MEDIUM severity      : {(sup['severity']=='MEDIUM').sum()} / {len(sup)}")
print(f"      (external driver: competitive marketing pressure on avg_roas DOWN)")

PASS  SUPPRESSED rows          : 6 / 6
PASS  llm_enhanced = False     : 6 / 6
PASS  escalation_suppressed    : 6 / 6
PASS  All MEDIUM severity      : 6 / 6
      (external driver: competitive marketing pressure on avg_roas DOWN)


---
## Test 6 — MONITOR Routing Integrity
All 74 Tier 3 MONITOR anomalies receive deterministic playbook text only — no Claude API calls.
These are daily-digest items that do not justify real-time LLM inference cost.

In [7]:
mon = df[df["layer4_priority_flag"] == "MONITOR"]

assert len(mon) == 74,                  f"Expected 74 MONITOR rows, got {len(mon)}"
assert (~mon["llm_enhanced"]).all(),    "Some MONITOR rows received LLM calls"

print(f"PASS  MONITOR rows         : {len(mon)} / 74")
print(f"PASS  llm_enhanced = False : {(~mon['llm_enhanced']).sum()} / {len(mon)}  (playbook text only)")
print(f"PASS  Tier distribution    : {dict(mon['tier'].value_counts().sort_index())}")

PASS  MONITOR rows         : 74 / 74
PASS  llm_enhanced = False : 74 / 74  (playbook text only)
PASS  Tier distribution    : {3: 74}


---
## Test 7 — Black Friday Spot-Check
The 2024-11-29 `total_revenue_usd` spike (+223.8%) is the highest-priority anomaly in the
dataset. It must rank #1, score in the HIGH band, and have a **negative** `revenue_at_risk`
(captured upside — money gained above forecast, not at risk).

In [8]:
bf = df[(df["date"] == "2024-11-29") & (df["kpi"] == "total_revenue_usd")].iloc[0]

assert int(bf["priority_rank"]) == 1,       f"Expected rank=1, got {bf['priority_rank']}"
assert bf["priority_band"] == "HIGH",        f"Expected HIGH, got {bf['priority_band']}"
assert float(bf["revenue_at_risk"]) < 0,    "Expected negative revenue_at_risk (captured upside)"
assert bf["llm_enhanced"],                  "Expected llm_enhanced=True for ESCALATE row"

print(f"PASS  anomaly_id      : {bf['anomaly_id']}")
print(f"PASS  priority_rank   : #{int(bf['priority_rank'])}  (highest priority in dataset)")
print(f"PASS  priority_band   : {bf['priority_band']}  (score={bf['priority_score']:.4f})")
print(f"PASS  revenue_at_risk : ${bf['revenue_at_risk']:,.0f}  (negative = captured upside)")
print(f"PASS  monthly_uplift  : ${bf['monthly_shortfall']:,.0f}")
print(f"PASS  deviation_pct   : {bf['deviation_pct']:+.1f}%")
print(f"PASS  llm_enhanced    : {bf['llm_enhanced']}")

PASS  anomaly_id      : ANO-20241129-REV
PASS  priority_rank   : #1  (highest priority in dataset)
PASS  priority_band   : HIGH  (score=0.9911)
PASS  revenue_at_risk : $-319,977  (negative = captured upside)
PASS  monthly_uplift  : $-1,371,330
PASS  deviation_pct   : +223.8%
PASS  llm_enhanced    : True


---
## Test 8 — Inventory Stockout Spot-Check
The 2024-03-15 `n_orders` anomaly (-35.7%) is the canonical inventory stockout event.
It must have positive `revenue_at_risk` (orders below forecast = revenue at risk) and
a matched playbook entry.

In [9]:
sk = df[(df["date"] == "2024-03-15") & (df["kpi"] == "n_orders")].iloc[0]

assert float(sk["revenue_at_risk"]) > 0,   "Expected positive revenue_at_risk (at risk)"
assert sk["playbook_match"],                "Expected a playbook match for this anomaly"
assert sk["llm_enhanced"],                  "Expected llm_enhanced=True for INVESTIGATE row"

print(f"PASS  anomaly_id      : {sk['anomaly_id']}")
print(f"PASS  priority_rank   : #{int(sk['priority_rank'])}")
print(f"PASS  priority_band   : {sk['priority_band']}  (score={sk['priority_score']:.4f})")
print(f"PASS  revenue_at_risk : ${sk['revenue_at_risk']:,.0f}  (positive = at risk)")
print(f"PASS  playbook_key    : {sk['playbook_key']}")
print(f"PASS  llm_enhanced    : {sk['llm_enhanced']}")
print(f"PASS  immediate_action: {str(sk['immediate_action'])[:90]}...")

PASS  anomaly_id      : ANO-20240315-ORD
PASS  priority_rank   : #53
PASS  priority_band   : HIGH  (score=0.8319)
PASS  revenue_at_risk : $27,888  (positive = at risk)
PASS  playbook_key    : order_volume_drop
PASS  llm_enhanced    : True
PASS  immediate_action: Audit checkout funnel (cart->payment->confirmation) in GA4; flag payment gateway error...


---
## Test 9 — Margin Impact Parity
`margin_impact` must equal `revenue_at_risk × avg_gross_margin` for every row.
Rounding to 2 decimal places in Step 4.1 introduces a small floating-point delta;
the maximum allowed deviation is 0.01.

In [10]:
expected_margin = df["revenue_at_risk"] * AVG_GROSS_MARGIN
delta           = (df["margin_impact"] - expected_margin).abs()
max_delta       = delta.max()

assert max_delta < 0.01, f"Max margin parity delta {max_delta:.6f} exceeds 0.01"

print(f"PASS  margin_impact = revenue_at_risk x {AVG_GROSS_MARGIN:.6f}")
print(f"PASS  Max absolute delta : {max_delta:.6f}  (threshold: 0.01)")
print(f"PASS  Mean absolute delta: {delta.mean():.6f}")

PASS  margin_impact = revenue_at_risk x 0.496782
PASS  Max absolute delta : 0.007159  (threshold: 0.01)
PASS  Mean absolute delta: 0.001568


---
## Test 10 — LLM Recommendation Coverage
All 101 ESCALATE + non-suppressed INVESTIGATE rows received Claude API calls.
Every LLM-enhanced row must have non-empty text in all three recommendation fields.

In [11]:
llm = df[df["llm_enhanced"]]

empty_imm = (llm["immediate_action"].fillna("").str.strip() == "").sum()
empty_st  = (llm["short_term_fix"].fillna("").str.strip() == "").sum()
empty_pr  = (llm["preventive_measure"].fillna("").str.strip() == "").sum()

assert len(llm) == 101,    f"Expected 101 LLM-enhanced rows, got {len(llm)}"
assert empty_imm == 0,     f"{empty_imm} rows have empty immediate_action"
assert empty_st  == 0,     f"{empty_st} rows have empty short_term_fix"
assert empty_pr  == 0,     f"{empty_pr} rows have empty preventive_measure"

print(f"PASS  LLM-enhanced rows              : {len(llm)} / 181")
print(f"PASS  Non-empty immediate_action     : {len(llm) - empty_imm} / {len(llm)}")
print(f"PASS  Non-empty short_term_fix       : {len(llm) - empty_st} / {len(llm)}")
print(f"PASS  Non-empty preventive_measure   : {len(llm) - empty_pr} / {len(llm)}")
print()
print("LLM routing breakdown:")
print(df.groupby(["layer4_priority_flag", "llm_enhanced"])
       .size().rename("count").reset_index().to_string(index=False))

PASS  LLM-enhanced rows              : 101 / 181
PASS  Non-empty immediate_action     : 101 / 101
PASS  Non-empty short_term_fix       : 101 / 101
PASS  Non-empty preventive_measure   : 101 / 101

LLM routing breakdown:
layer4_priority_flag  llm_enhanced  count
            ESCALATE         False      0
            ESCALATE          True     15
         INVESTIGATE         False      0
         INVESTIGATE          True     86
             MONITOR         False     74
             MONITOR          True      0
          SUPPRESSED         False      6
          SUPPRESSED          True      0


---
## Test 11 — Effort Level Values
`effort_level` must only contain the values H (high — multi-day cross-team), M (medium —
same-day task), or L (low — under 1-hour check). No nulls or unexpected values.

In [12]:
valid = {"H", "M", "L"}
found = set(df["effort_level"].unique())

assert found.issubset(valid), f"Unexpected effort values: {found - valid}"
assert df["effort_level"].notna().all(), "effort_level contains nulls"

counts = df["effort_level"].value_counts()
print(f"PASS  effort_level values: {sorted(found)}  (no unexpected values, no nulls)")
for effort in ["H", "M", "L"]:
    n = counts.get(effort, 0)
    print(f"  {effort}  {n:>3}  ({n/181*100:.1f}%)")

PASS  effort_level values: ['H', 'L', 'M']  (no unexpected values, no nulls)
  H    3  (1.7%)
  M   60  (33.1%)
  L  118  (65.2%)


---
## Test 12 — SQLite Parity
The `intelligence_results` table in `kpi_anomaly_detection.db` must have exactly 181 rows,
and all 14 Layer 1–4 tables must be present and match their expected row counts.

In [13]:
conn = sqlite3.connect("../data/kpi_anomaly_detection.db")

expected_tables = {
    "processed_kpis"        :   731,
    "method_a_results"      :  8772,
    "method_b_results"      :   731,
    "method_c_results"      :  2924,
    "anomaly_results"       :   181,
    "ensemble_voting_matrix":  8772,
    "rca_graph_results"     :   181,
    "rca_causal_results"    :   181,
    "rca_results"           :   181,
    "rca_assembly"          :   181,
    "impact_results"        :   181,
    "priority_results"      :   181,
    "recommendations"       :   181,
    "intelligence_results"  :   181,
}

db_tables = [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()]

for table, expected_n in expected_tables.items():
    assert table in db_tables, f"Missing table: {table}"
    actual_n = conn.execute(f"SELECT COUNT(*) FROM [{table}]").fetchone()[0]
    assert actual_n == expected_n, f"{table}: expected {expected_n}, got {actual_n}"
    print(f"PASS  {table:<30}  {actual_n:>6,} rows")

conn.close()

PASS  processed_kpis                   731 rows
PASS  method_a_results               8,772 rows
PASS  method_b_results                 731 rows
PASS  method_c_results               2,924 rows
PASS  anomaly_results                  181 rows
PASS  ensemble_voting_matrix          8,772 rows
PASS  rca_graph_results                181 rows
PASS  rca_causal_results               181 rows
PASS  rca_results                      181 rows
PASS  rca_assembly                     181 rows
PASS  impact_results                   181 rows
PASS  priority_results                 181 rows
PASS  recommendations                  181 rows
PASS  intelligence_results             181 rows


---
## All 12 Tests Summary

| # | Test | Expected |
|---|---|---|
| T01 | Shape | (181, 68) |
| T02 | Priority score range | All in [0, 1], 0 nulls |
| T03 | Priority rank uniqueness | 181 unique integers 1–181 |
| T04 | ESCALATE → HIGH band | 15/15 ESCALATE rows in HIGH |
| T05 | SUPPRESSED routing | 6 rows, llm_enhanced=False, escalation_suppressed=True |
| T06 | MONITOR routing | 74 rows, llm_enhanced=False |
| T07 | Black Friday spot-check | rank=1, HIGH, revenue_at_risk < 0 |
| T08 | Stockout spot-check | revenue_at_risk > 0, playbook matched, llm_enhanced=True |
| T09 | Margin impact parity | max delta < 0.01 |
| T10 | LLM recommendation coverage | 101/101 rows with non-empty actions |
| T11 | Effort level values | Only H / M / L |
| T12 | SQLite parity | 14 tables, all row counts correct |

In [14]:
print("All 12 Layer 4 quality tests passed.")
print("intelligence_results.csv is certified ready for Layer 5 (Communication Layer).")

All 12 Layer 4 quality tests passed.
intelligence_results.csv is certified ready for Layer 5 (Communication Layer).
